# 00 模型为什么能识图：从像素到视觉 token，再到文字回答

目标：用一个可运行的小型多模态模型拆开识图链路，再用 Gemma 4 对比更大的原生多模态模型。

| 层级 | ModelScope 模型 ID | 本教程用途 |
| --- | --- | --- |
| 小模型 | `Qwen/Qwen2.5-VL-3B-Instruct` | 默认实跑，观察图片输入张量、视觉 token、内部模块和生成过程 |
| 大模型 | `google/gemma-4-E4B-it` | 可选实验，对比 Gemma 4 的统一多模态、可变视觉 token budget 和长上下文 |

Gemma 4 E4B 的 BF16 权重约 16GB，实际运行还需要 KV cache 和中间激活，建议至少 24GB 显存。默认先把 `RUN_GEMMA4` 保持为 `False`。

## 1. 先建立完整心智模型

视觉语言模型通常不是直接理解 JPEG 文件，而是把图片转换成一串能被语言模型处理的向量：

```text
JPEG / PNG 文件
-> 解码为 RGB 像素
-> resize、normalize、切成 patches
-> Vision Encoder 提取视觉特征
-> Projector / Merger 映射到语言模型 hidden size
-> 视觉向量与文字 token embedding 一起进入语言模型
-> 自回归预测下一个文字 token
```

模型之所以能“识图”，不是因为它保存了一套人工规则，而是因为训练阶段看过大量图片与文字的对应关系，并通过损失函数学会：哪些视觉模式与哪些语言概念相关。

## 2. 安装依赖

Gemma 4 需要较新的 Transformers。若升级后提示重启内核，请重启再继续运行。

In [ ]:
from pathlib import Path

requirements_path = Path("vision/requirements-vision.txt")
if not requirements_path.exists():
    requirements_path = Path("requirements-vision.txt")

print("requirements:", requirements_path)
%pip install -U -r {requirements_path}

## 3. 配置模型源、设备和运行开关

ModelScope Notebook 默认从魔搭下载模型。本地已有模型目录时，也可以直接把模型 ID 改成目录路径。

AMD ROCm 环境中，PyTorch 仍通过 `torch.cuda` 和 `cuda:0` 兼容接口访问 AMD GPU，这是正常现象。

In [ ]:
import gc
import os
from pathlib import Path

import torch

SMALL_MODEL_ID = os.getenv("SMALL_VLM_ID", "Qwen/Qwen2.5-VL-3B-Instruct")
GEMMA4_MODEL_ID = os.getenv("GEMMA4_MODEL_ID", "google/gemma-4-E4B-it")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()

RUN_SMALL_MODEL = True
RUN_GEMMA4 = False

if torch.cuda.is_available():
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    DTYPE = torch.float32

def resolve_model_path(model_id):
    if Path(model_id).exists() or MODEL_SOURCE != "modelscope":
        return model_id
    from modelscope import snapshot_download
    return snapshot_download(model_id)

print("torch:", torch.__version__)
print("ROCm/HIP:", torch.version.hip)
print("CUDA runtime:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
print("dtype:", DTYPE)

## 4. 本地生成一张教学图片

我们故意在图片里放入颜色、形状、数量、柱状图和文字。这样可以分别测试：物体识别、空间理解、计数、图表理解和 OCR。

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

output_dir = Path("vision/outputs") if Path("vision").exists() else Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)
image_path = (output_dir / "vlm_teaching_image.png").resolve()

image = Image.new("RGB", (840, 560), "white")
draw = ImageDraw.Draw(image)
def load_font(size):
    for font_name in ("DejaVuSans.ttf", "Arial.ttf"):
        try:
            return ImageFont.truetype(font_name, size=size)
        except OSError:
            pass
    return ImageFont.load_default()

font = load_font(28)
small_font = load_font(20)

draw.text((35, 25), "VISION LAB", fill="black", font=font)
draw.ellipse((70, 120, 250, 300), fill="#e74c3c", outline="black", width=4)
draw.text((110, 315), "RED CIRCLE", fill="black", font=small_font)
draw.rectangle((330, 120, 590, 300), fill="#3498db", outline="black", width=4)
draw.text((375, 315), "BLUE BOX", fill="black", font=small_font)

bar_x = [85, 180, 275]
bar_h = [70, 130, 200]
bar_colors = ["#2ecc71", "#f1c40f", "#9b59b6"]
for index, (x, height, color) in enumerate(zip(bar_x, bar_h, bar_colors), start=1):
    draw.rectangle((x, 520 - height, x + 55, 520), fill=color, outline="black")
    draw.text((x + 18, 525), str(index), fill="black", font=small_font)

draw.text((430, 405), "Total: 42", fill="black", font=font)
draw.text((430, 460), "Question: which bar is tallest?", fill="black", font=small_font)
image.save(image_path)

print(image_path)
display(image)

## 5. 加载小模型：Qwen2.5-VL-3B

`AutoProcessor` 同时承担文本 tokenizer 和图片预处理工作。模型类负责接收文本 token 与视觉张量，并生成文字回答。

In [ ]:
if RUN_SMALL_MODEL:
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

    small_model_path = resolve_model_path(SMALL_MODEL_ID)
    processor = AutoProcessor.from_pretrained(small_model_path, trust_remote_code=True)
    small_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        small_model_path,
        dtype=DTYPE,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    ).eval()

    small_device = next(small_model.parameters()).device
    parameter_count = sum(parameter.numel() for parameter in small_model.parameters())
    print("model path:", small_model_path)
    print("parameters:", f"{parameter_count / 1e9:.2f}B")
    print("first parameter device:", small_device)
    print("first parameter dtype:", next(small_model.parameters()).dtype)

## 6. 图片真正进入模型时是什么

对 Qwen2.5-VL 来说，processor 通常会产生：

- `input_ids`：文字 token，以及表示图片位置的特殊 token。
- `attention_mask`：标记哪些位置有效。
- `pixel_values`：图片经过 resize、normalize、patchify 后的数值张量，不再是 PNG/JPEG 字节。
- `image_grid_thw`：图片视觉网格的时间、高度、宽度信息；单图的时间维通常为 1。

注意：不同 VLM 的字段名和张量形状不完全相同，不能假设所有模型都有 `pixel_values` 或 `image_grid_thw`。

In [ ]:
if RUN_SMALL_MODEL:
    from qwen_vl_utils import process_vision_info

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path.as_uri()},
                {"type": "text", "text": "请描述图片，并回答哪根柱子最高、图片中的数字是多少。"},
            ],
        }
    ]

    rendered_text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    image_inputs, video_inputs = process_vision_info(messages)
    small_inputs = processor(
        text=[rendered_text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(small_device)

    print("rendered prompt:\n", rendered_text)
    print("\ninput fields:")
    for key, value in small_inputs.items():
        print(f"{key:18s} shape={tuple(value.shape)}, dtype={value.dtype}, device={value.device}")

### 如何理解这些形状

```text
input_ids       [batch, sequence_length]
attention_mask  [batch, sequence_length]
pixel_values    Qwen2.5-VL 已切好的视觉 patches
image_grid_thw  [image_count, 3]，每行是 temporal / height / width 网格
```

关键区别：文本最初是离散 token ID；图片最初是连续像素值。二者经过各自编码器后，最终都会变成形如 `[token_count, hidden_size]` 的向量，语言模型才能把它们放进同一个上下文中处理。

In [ ]:
if RUN_SMALL_MODEL:
    token_ids = small_inputs["input_ids"][0].detach().cpu().tolist()
    tokens = processor.tokenizer.convert_ids_to_tokens(token_ids)
    image_marker_tokens = [token for token in tokens if "image" in token or "vision" in token]

    print("text sequence length:", len(token_ids))
    print("image/vision marker count:", len(image_marker_tokens))
    print("first marker tokens:", image_marker_tokens[:20])
    print("image_grid_thw:", small_inputs["image_grid_thw"].detach().cpu().tolist())

## 7. 让模型回答图片问题

识图模型最后仍然是通过 `generate()` 一个 token 一个 token 地生成文字。图片主要影响 prefill 阶段形成的上下文表示，之后 decode 阶段继续使用 KV cache。

In [ ]:
if RUN_SMALL_MODEL:
    with torch.inference_mode():
        generated_ids = small_model.generate(
            **small_inputs,
            max_new_tokens=160,
            do_sample=False,
        )

    new_ids = generated_ids[:, small_inputs["input_ids"].shape[1]:]
    answer = processor.batch_decode(
        new_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
    print(answer)

## 8. 模型内部架构是什么样的

典型视觉语言模型可以拆成三部分：

1. **Vision Encoder**：通常是 ViT，把图片 patches 编码成视觉特征。
2. **Projector / Merger**：压缩视觉 token，并把视觉维度映射到语言模型 hidden size。
3. **Language Model**：结合问题和视觉特征，预测文字答案。

Qwen2.5-VL 的视觉编码器使用动态分辨率、窗口注意力和视觉位置编码；语言侧仍然是 decoder-only Transformer。

In [ ]:
if RUN_SMALL_MODEL:
    print("model class:", type(small_model).__name__)
    print("config class:", type(small_model.config).__name__)
    print("vision config:", small_model.config.vision_config)
    print("text config:", small_model.config.text_config)

    print("\ninteresting module names:")
    keywords = ("visual", "vision", "merger", "projector", "language_model")
    for name, module in small_model.named_modules():
        if name and any(keyword in name.lower() for keyword in keywords):
            depth = name.count(".")
            if depth <= 3:
                print(f"{name:45s} -> {type(module).__name__}")

### 用 forward hook 观察视觉模块输出

hook 不修改模型，只在 forward 经过目标模块时记录输出形状。视觉编码器输出回答“图片被编码成多少个向量”，merger 输出回答“最终有多少视觉 token 交给语言模型”。

In [ ]:
if RUN_SMALL_MODEL:
    captured = {}

    def shape_of(value):
        if hasattr(value, "shape"):
            return tuple(value.shape)
        if isinstance(value, (tuple, list)):
            return [shape_of(item) for item in value[:3]]
        return type(value).__name__

    def capture(name):
        def hook(_module, _args, output):
            captured[name] = shape_of(output)
        return hook

    handles = []
    if hasattr(small_model, "visual"):
        handles.append(small_model.visual.register_forward_hook(capture("visual output")))
        if hasattr(small_model.visual, "merger"):
            handles.append(small_model.visual.merger.register_forward_hook(capture("merger output")))

    with torch.inference_mode():
        forward_outputs = small_model(**small_inputs, use_cache=True)

    for handle in handles:
        handle.remove()

    print("captured:", captured)
    print("logits:", tuple(forward_outputs.logits.shape))
    print("KV cache layers:", len(forward_outputs.past_key_values))

## 9. 视觉 token 是部署成本的关键

图片分辨率越高，通常会产生越多 patches 和视觉 token：

- Vision Encoder 计算量增加。
- Language Model 的 prefill 序列变长。
- attention 和 KV cache 成本增加。
- OCR、小目标和细节识别通常更好，但收益不会无限增长。

Qwen2.5-VL 会先形成视觉网格，再通过 spatial merge 压缩 token。下面从实际配置估算交给语言模型的视觉 token 数。

In [ ]:
if RUN_SMALL_MODEL:
    grid_thw = small_inputs["image_grid_thw"].detach().cpu()
    merge_size = getattr(small_model.config.vision_config, "spatial_merge_size", 1)
    raw_patch_count = int(grid_thw.prod(dim=1).sum().item())
    estimated_visual_tokens = raw_patch_count // (merge_size ** 2)

    print("image_grid_thw:", grid_thw.tolist())
    print("raw patch count:", raw_patch_count)
    print("spatial merge size:", merge_size)
    print("estimated visual tokens after merge:", estimated_visual_tokens)

### 动态分辨率实验：同一张图，不同视觉 token 数

这个实验只运行 processor，不重复执行大模型生成，因此速度较快。实际生产中可按任务限制 `min_pixels / max_pixels` 或指定 resize 尺寸。

In [ ]:
if RUN_SMALL_MODEL:
    def inspect_resolution(height, width):
        resolution_messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": image_path.as_uri(),
                        "resized_height": height,
                        "resized_width": width,
                    },
                    {"type": "text", "text": "描述图片。"},
                ],
            }
        ]
        text = processor.apply_chat_template(resolution_messages, tokenize=False, add_generation_prompt=True)
        images, videos = process_vision_info(resolution_messages)
        batch = processor(text=[text], images=images, videos=videos, return_tensors="pt")
        grid = batch["image_grid_thw"]
        patch_count = int(grid.prod(dim=1).sum().item())
        visual_tokens = patch_count // (merge_size ** 2)
        return tuple(grid[0].tolist()), visual_tokens, tuple(batch["pixel_values"].shape)

    for height, width in [(280, 420), (560, 840), (840, 1260)]:
        grid, visual_tokens, pixel_shape = inspect_resolution(height, width)
        print(f"resize={height}x{width} -> grid={grid}, visual_tokens~{visual_tokens}, pixel_values={pixel_shape}")

## 10. 图片真的影响了预测吗：对照实验

保持问题和图片尺寸不变，只把真实图片替换为空白图片，然后比较“下一个 token”的 logits。若两个概率分布不同，说明视觉内容确实改变了模型判断。

这不是完整的可解释性方法，但比只看最终答案更接近模型内部行为。

In [ ]:
if RUN_SMALL_MODEL:
    import torch.nn.functional as F

    blank_path = (output_dir / "blank_same_size.png").resolve()
    Image.new("RGB", image.size, "white").save(blank_path)

    def next_token_logits(path):
        compare_messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": path.as_uri()},
                    {"type": "text", "text": "图片中的主要颜色和数字是什么？"},
                ],
            }
        ]
        text = processor.apply_chat_template(compare_messages, tokenize=False, add_generation_prompt=True)
        images, videos = process_vision_info(compare_messages)
        batch = processor(text=[text], images=images, videos=videos, return_tensors="pt").to(small_device)
        with torch.inference_mode():
            logits = small_model(**batch, use_cache=False).logits[:, -1, :].float()
        return logits

    real_logits = next_token_logits(image_path)
    blank_logits = next_token_logits(blank_path)
    cosine = F.cosine_similarity(real_logits, blank_logits).item()
    mean_abs_delta = (real_logits - blank_logits).abs().mean().item()

    print("logits cosine similarity:", round(cosine, 6))
    print("mean absolute logits delta:", round(mean_abs_delta, 6))

## 11. 模型是怎样学会图片和文字对应关系的

常见训练过程可以概括为：

1. **视觉预训练**：Vision Encoder 从大量图片中学习边缘、纹理、物体、文字和空间关系。
2. **跨模态对齐**：训练 projector/merger，让视觉特征进入语言模型后能对应文字概念。
3. **多模态预训练**：使用图文对、OCR、文档、图表、定位、视频等数据做 next-token prediction。
4. **指令微调**：训练模型按人类问题回答，而不只是生成图片描述。
5. **偏好与安全对齐**：改善回答质量、拒答策略和风险控制。

核心损失仍经常是文字 token 的交叉熵：给定图片和前文，预测正确的下一个回答 token。视觉编码器收到的梯度会推动它提取对回答有帮助的特征。

## 12. 进阶原理

### 12.1 Prefix visual tokens 与 cross-attention

- **Prefix / unified token 路线**：把视觉特征映射成类似文本 embedding 的向量，放入语言模型上下文。实现统一，视觉 token 会增加上下文长度。
- **Cross-attention 路线**：语言模型在部分层通过单独的 cross-attention 读取视觉特征。视觉与文本流更独立，但架构更复杂。

### 12.2 Patch 不等于物体

ViT 通常先按固定小块切图。一个 patch 可能只包含物体的一部分；“圆形”“柱状图”等高级概念是在多层 attention 中组合出来的。

### 12.3 位置编码非常重要

只知道 patch 内容，不知道它位于左上还是右下，模型就难以理解布局。多模态位置编码需要表达二维空间；视频还需要表达时间顺序。

### 12.4 分辨率是质量与成本旋钮

低分辨率适合分类和粗粒度描述；OCR、文档、小目标和精细定位通常需要更多视觉 token。生产环境应按任务设置视觉 token budget，而不是无条件使用最高分辨率。

## 13. 估算视觉 token 带来的 KV cache 成本

对把视觉 token 放入语言模型上下文的架构，可用下面公式粗估它们带来的 KV cache：

```text
KV bytes ≈ 2 × layers × KV heads × head_dim × token_count × dtype_bytes
```

`2` 表示 Key 和 Value。这个估算不包括 Vision Encoder 激活、attention 临时张量、模型权重和框架开销。

In [ ]:
if RUN_SMALL_MODEL:
    text_config = small_model.config.text_config
    layers = text_config.num_hidden_layers
    attention_heads = text_config.num_attention_heads
    kv_heads = getattr(text_config, "num_key_value_heads", attention_heads)
    head_dim = getattr(text_config, "head_dim", text_config.hidden_size // attention_heads)
    dtype_bytes = next(small_model.parameters()).element_size()

    def estimate_kv_mib(token_count):
        byte_count = 2 * layers * kv_heads * head_dim * token_count * dtype_bytes
        return byte_count / 1024 ** 2

    print("estimated visual-token KV cache:", f"{estimate_kv_mib(estimated_visual_tokens):.2f} MiB")
    for token_count in [256, 512, 1024, 2048]:
        print(f"{token_count:4d} extra tokens -> {estimate_kv_mib(token_count):8.2f} MiB per request")

## 14. 可选大模型实验：Gemma 4 E4B

ModelScope 模型 ID 已确认：`google/gemma-4-E4B-it`。

Gemma 4 E4B 支持文本、图片和音频输入，采用新一代原生多模态架构，并支持可变视觉 token budget。它名为 E4B，但由于 per-layer embeddings 等参数，BF16 权重文件约 16GB。

运行前建议：

- 确认至少约 24GB 可用显存。
- 将前面的 `RUN_GEMMA4 = False` 改为 `True`。
- 若显存不足，重启内核后只运行配置、图片和本节，跳过 Qwen 小模型。

In [ ]:
if RUN_GEMMA4:
    from transformers import AutoModelForMultimodalLM, AutoProcessor

    gemma4_path = resolve_model_path(GEMMA4_MODEL_ID)
    gemma4_processor = AutoProcessor.from_pretrained(gemma4_path, trust_remote_code=True)
    gemma4_model = AutoModelForMultimodalLM.from_pretrained(
        gemma4_path,
        dtype=DTYPE,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    ).eval()

    gemma4_device = next(gemma4_model.parameters()).device
    print("Gemma 4 device:", gemma4_device)
    print("Gemma 4 config:", gemma4_model.config)

In [ ]:
if RUN_GEMMA4:
    gemma_messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "url": image_path.as_uri()},
                {"type": "text", "text": "请描述图片，并回答哪根柱子最高、图片中的数字是多少。"},
            ],
        }
    ]

    gemma_inputs = gemma4_processor.apply_chat_template(
        gemma_messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(gemma4_device)

    print("Gemma 4 input fields:")
    for key, value in gemma_inputs.items():
        if hasattr(value, "shape"):
            print(f"{key:18s} shape={tuple(value.shape)}, dtype={value.dtype}")

    input_length = gemma_inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        gemma_output = gemma4_model.generate(**gemma_inputs, max_new_tokens=160, do_sample=False)

    raw_response = gemma4_processor.decode(gemma_output[0][input_length:], skip_special_tokens=False)
    if hasattr(gemma4_processor, "parse_response"):
        print(gemma4_processor.parse_response(raw_response))
    else:
        print(raw_response)

## 15. Qwen2.5-VL 与 Gemma 4 应该怎样比较

| 观察维度 | Qwen2.5-VL-3B | Gemma 4 E4B |
| --- | --- | --- |
| 教学价值 | 结构成熟，processor 输入字段容易观察 | 原生多模态更广，适合观察新架构能力 |
| 图片成本控制 | dynamic resolution、`min_pixels/max_pixels` | 可变视觉 token budget |
| 模态 | 图像、视频、文本 | 图像、音频、文本；视频可按帧处理 |
| 权重规模 | 约 3B 语言模型级别，较容易实跑 | E4B 有额外 embedding 等参数，BF16 权重约 16GB |
| 适合场景 | 文档、OCR、定位、图表、视频理解 | 统一多模态、长上下文、推理和端侧方向 |

不要只比较参数量。实际部署还应比较：目标任务准确率、视觉 token 数、首 token 延迟、吞吐、峰值显存、输入分辨率和框架支持。

## 16. 识图模型常见风险与评估方法

- **视觉幻觉**：图片里没有某物，模型却根据语言先验编造出来。加入“看不到就明确说看不到”的提示只能缓解，不能彻底解决。
- **OCR 与小目标失败**：文字太小、压缩严重、旋转或遮挡时容易识别错误。应测试不同分辨率和裁剪策略。
- **计数不稳定**：视觉 token 和 attention 并不天然等价于精确检测器，密集物体计数容易出错。
- **提示词偏置**：诱导性问题可能让模型迎合错误前提。应加入反事实和空白图对照测试。
- **隐私与安全**：图片可能包含人脸、证件、位置、屏幕内容和隐私文本，上传与日志保存需要合规控制。
- **高分辨率拒绝服务风险**：超大图片、超多图片或视频帧会制造大量视觉 token，服务端必须限制尺寸、帧数和总 token budget。

推荐评估集至少覆盖：普通图片描述、OCR、图表、文档、小目标、空间关系、计数、空白图、模糊图、诱导问题和领域内真实图片。

## 17. 面试记忆线

可以这样回答“多模态大模型为什么能识图”：

> 图片先由 processor 解码、缩放、归一化并切成 patches；Vision Transformer 把 patches 编码为视觉特征，projector 或 merger 再把它们映射到语言模型的 hidden space。模型在大规模图文对、多模态预训练和指令微调中，通过 next-token loss 学会视觉模式与文字概念的对应关系，因此能结合图片和问题生成答案。部署时视觉 token 数会直接影响 Vision Encoder 成本、prefill 延迟、KV cache 和显存，所以需要限制图片分辨率、数量和 token budget。

进一步追问时，应能解释：

- 像素、patch、视觉特征和视觉 token 的区别。
- Vision Encoder、projector/merger、LLM 各自负责什么。
- 动态分辨率为什么能提升细节能力，也为什么会增加成本。
- 视觉幻觉、OCR、小目标和计数为什么需要单独评估。

## 18. 建议继续完成的练习

1. 把真实图片换成手机截图，比较 OCR 在三档分辨率下的输出与耗时。
2. 对同一张图分别问中性问题和诱导问题，观察视觉幻觉。
3. 输入两张图片，让模型比较差异，并记录视觉 token 数增长。
4. 用 forward hook 继续观察视觉 block 与 merger 的输出形状。
5. 在相同图片和问题下，对比 Qwen2.5-VL 与 Gemma 4 的答案、首 token 延迟和峰值显存。

In [ ]:
# 可选：实验结束后释放显存
if "forward_outputs" in globals():
    del forward_outputs
if "small_inputs" in globals():
    del small_inputs
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("cleanup finished")